In [2]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "stage1-nested"   # the branch with the code this notebook needs (set back to "main" after merging)

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", url, CODE], check=True)
# fetch + checkout + pull, so a clone left on another branch earlier in the session switches over
subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "origin", BRANCH], check=True)
print(BRANCH, subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
_MINE = {"fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic",
         "stage1_structural", "nested_builder"}
for _m in list(sys.modules):
    if _m in _MINE or _m == "src" or _m.startswith("src."):   # src.* = the team's nested code
        sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
stage1-nested 9a30690 updated



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ammara-asif125 (kashaf-saleem09-institute-of-business-administration) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [3]:
import numpy as np, pandas as pd

def table(configs):
    """One row per config: accuracy metrics, cost, and per-level means over tasks (nested arms)."""
    rows = []
    for cfg in configs:
        base = {"arm": sr.arm_label(cfg), "lr": cfg["lr"], "seed": cfg["seed"]}
        r = load_result(cfg, paths)
        if r is None:
            rows.append({**base, "status": "not finished"}); continue
        m = r["metrics"]
        row = {**base, "run_id": r["run_id"], "avg_incremental_acc": m["avg_incremental_acc"],
               "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
               # mean of the diagonal: accuracy on each task right after learning it (plasticity check)
               "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
               "steps": m["optimizer_steps"], "n_params": m["n_params"],
               "minutes": r["wall_clock_s"] / 60, "peak_gpu_mb": m.get("peak_gpu_mem_mb")}
        log = m.get("level_log_per_exp") or []
        for k in (log[0] if log else {}):
            if k.startswith(("updates/", "rel_displacement/", "deployed_rel_displacement/", "penalty_ratio/")):
                row[k] = float(np.mean([x[k] for x in log]))
        rows.append(row)
    return pd.DataFrame(rows)

In [4]:
EPOCHS = 10
SEEDS = [0, 1, 2, 3, 4]               # 0-2 exist and are skipped
ARMS = {"flat": 0.1, "nested_a": 0.5, "nested_b": 0.6}   # tuned lrs from validation (unchanged)
GATE1 = [sr.arm_cfg(name, stage="stage1_nested_control", epochs=EPOCHS, seed=s, lr=lr,
                    eval_split="test", save_checkpoints=True)
         for s in SEEDS for name, lr in ARMS.items()]   # seed outer: each seed finishes every arm first
for c in GATE1: print(run_id(c), sr.arm_label(c), c["lr"], c["seed"], "| done" if load_result(c, paths) else "")

07a97dd12ec3 flat_nl 0.1 0 | done
82f66588c5f1 nested_a 0.5 0 | done
ea0983312db5 nested_b 0.6 0 | done
78e133183087 flat_nl 0.1 1 | done
1a6e0b7da780 nested_a 0.5 1 | done
48523f3dfeea nested_b 0.6 1 | done
b09e3cc29bda flat_nl 0.1 2 | done
9da44b28d332 nested_a 0.5 2 | done
badf7f571542 nested_b 0.6 2 | done
97e56eaf8f25 flat_nl 0.1 3 
fee1f9d57dba nested_a 0.5 3 
3faea1c21912 nested_b 0.6 3 
a1e14e58bff7 flat_nl 0.1 4 
909b2256d515 nested_a 0.5 4 
f6544537e625 nested_b 0.6 4 


In [5]:
out = run_sweep(GATE1, sr.run_experiment, paths)

[1/15] 07a97dd12ec3 already finished, skipping
[2/15] 82f66588c5f1 already finished, skipping
[3/15] ea0983312db5 already finished, skipping
[4/15] 78e133183087 already finished, skipping
[5/15] 1a6e0b7da780 already finished, skipping
[6/15] 48523f3dfeea already finished, skipping
[7/15] b09e3cc29bda already finished, skipping
[8/15] 9da44b28d332 already finished, skipping
[9/15] badf7f571542 already finished, skipping
[10/15] 97e56eaf8f25 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 80.20%  current 80.20%  train-loss 0.817
  exp 2/20: seen-avg 38.10%  current 76.20%  train-loss 0.851
  exp 3/20: seen-avg 25.13%  current 75.40%  train-loss 0.701
  exp 4/20: seen-avg 18.40%  current 73.60%  train-loss 0.686
  exp 5/20: seen-avg 16.36%  current 81.80%  train-loss 0.763
  exp 6/20: seen-avg 13.77%  current 82.60%  train-loss 0.782
  exp 7/20: seen-avg 12.14%  current 85.00%  train-loss 0.593
  exp 8/20: seen-avg 11.05%  current 88.40%  train-loss 0.546
  exp 9/20: seen-avg 10.49%  current 94.40%  train-loss 0.422
  exp 10/20: seen-avg 9.20%  current 92.00%  train-loss 0.422
  exp 11/20: seen-avg 6.98%  current 76.80%  train-loss 0.670
  exp 12/20: seen-avg 7.55%  current 90.60%  train-loss 0.510
  exp 13/20: seen-avg 5.68%  current 73.80%  train-loss 0.716
  exp 14/20: seen-avg 5.74%  current 80.40%  train-loss 0.763
  exp 15/20: seen-avg 5.99%  current 89.80%  train-loss 0.502
  exp 16/20: seen-avg 4.65%  current 74.40%  train-loss 0.783
  exp 17

acc_current_exp,▃▂▂▁▄▄▅▆█▇▂▇▁▃▆▁▇▄▆▅
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[11/15] fee1f9d57dba running


...

acc_current_exp,▁▁▅▅▂▆▂▇██▆▆▂▅▆▅▄█▇█
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[15/15] f6544537e625 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 72.00%  current 72.00%  train-loss 0.795
  exp 2/20: seen-avg 38.00%  current 76.00%  train-loss 0.584
  exp 3/20: seen-avg 29.67%  current 89.00%  train-loss 0.529
  exp 4/20: seen-avg 21.65%  current 86.60%  train-loss 0.521
  exp 5/20: seen-avg 15.68%  current 78.40%  train-loss 0.696
  exp 6/20: seen-avg 14.47%  current 86.80%  train-loss 0.581
  exp 7/20: seen-avg 10.91%  current 76.40%  train-loss 0.742
  exp 8/20: seen-avg 11.20%  current 89.60%  train-loss 0.443
  exp 9/20: seen-avg 10.36%  current 93.20%  train-loss 0.402
  exp 10/20: seen-avg 9.10%  current 91.00%  train-loss 0.437
  exp 11/20: seen-avg 7.85%  current 86.40%  train-loss 0.501
  exp 12/20: seen-avg 7.53%  current 90.40%  train-loss 0.441
  exp 13/20: seen-avg 6.02%  current 78.20%  train-loss 0.697
  exp 14/20: seen-avg 5.97%  current 83.60%  train-loss 0.570
  exp 15/20: seen-avg 5.59%  current 83.80%  train-loss 0.504
  exp 16/20: seen-avg 5.19%  current 83.00%  train-loss 0.611
  exp 17

acc_current_exp,▁▂▇▆▃▆▂▇█▇▆▇▃▅▅▅▄▇▆█
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


In [6]:
cdf = table(GATE1)
done = cdf[cdf["status"] != "not finished"] if "status" in cdf else cdf
assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
cols = ["avg_incremental_acc", "acc_just_learned", "final_avg_acc", "avg_forgetting", "minutes"]
display(done.groupby(["arm", "lr"])[cols].agg(["mean", "std", "count"]).round(2))

avg_incremental_acc             acc_just_learned              \
                            mean   std count             mean   std count   
arm      lr                                                                 
flat_nl  0.1               14.32  0.32     5            82.65  0.75     5   
nested_a 0.5               14.19  0.30     5            83.43  0.40     5   
nested_b 0.6               14.27  0.38     5            82.59  1.41     5   

             final_avg_acc             avg_forgetting             minutes  \
                      mean   std count           mean   std count    mean   
arm      lr                                                                 
flat_nl  0.1          4.29  0.27     5          82.49  0.67     5    8.78   
nested_a 0.5          4.36  0.20     5          83.24  0.42     5    8.63   
nested_b 0.6          4.28  0.32     5          82.42  1.45     5    8.55   

                          
               std count  
arm      lr               
flat_nl  0.1  0.17     5  
nested_a 0.5  0.13     5  
nested_b 0.6  0.16     5